In [1]:
import os
import json
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer
from sentence_transformers import CrossEncoder
from tqdm import tqdm

QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
QDRANT_COLLECTION = os.getenv("QDRANT_COLLECTION", "tamanh-hospital-v1")

EMBEDDING_MODEL = os.getenv(
    "EMBEDDING_MODEL",
    "bkai-foundation-models/vietnamese-bi-encoder"
)
RERANK_MODEL = os.getenv(
    "RERANK_MODEL",
    "itdainb/PhoRanker"
)

INPUT_QUERIES = "./eval_queries.jsonl"
OUTPUT_RESULTS = "./eval_search_results.jsonl"
TOP_K_SEARCH = 10
TOP_K_RERANK = 5
RERANK_SCORE_THRESHOLD = 0.4

def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records

def save_jsonl(path, records):
    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

def build_query_embedding(model, text):
    emb = model.encode(text, convert_to_numpy=True)
    return emb.tolist()

def main():
    queries = load_jsonl(INPUT_QUERIES)
    if not queries:
        raise ValueError(f"No queries found in {INPUT_QUERIES}")

    print("[Info] Load embedding model:", EMBEDDING_MODEL)
    embedding_model = SentenceTransformer(EMBEDDING_MODEL, device="cuda" if os.getenv("CUDA_VISIBLE_DEVICES", None) else "cpu")

    print("[Info] Load rerank model:", RERANK_MODEL)
    rerank_model = CrossEncoder(RERANK_MODEL, max_length=256)

    client = QdrantClient(url=QDRANT_URL)

    all_results = []

    for query in tqdm(queries, desc="Queries"):
        query_id = query["id"]
        query_text = query["text"]

        query_vector = build_query_embedding(embedding_model, query_text)

        search_hits = client.search(
            collection_name=QDRANT_COLLECTION,
            query_vector=query_vector,
            limit=TOP_K_SEARCH,
            with_payload=True,
        )

        docs = []
        seen_text = set()
        for hit in search_hits:
            payload = hit.payload or {}
            chunk_text = payload.get("chunk", "")
            if not isinstance(chunk_text, str) or not chunk_text.strip():
                continue

            chunk_id = hit.id
            # Qdrant id có thể trả về int hoặc UUID; ở bạn hiện dùng int
            if isinstance(chunk_id, str):
                try:
                    chunk_id = int(chunk_id)
                except ValueError:
                    pass

            if chunk_text in seen_text:
                continue
            seen_text.add(chunk_text)

            docs.append({
                "chunk_id": chunk_id,
                "chunk": chunk_text,
                "score": float(hit.score) if hit.score is not None else None
            })

        if not docs:
            continue

        rerank_inputs = [[query_text, doc["chunk"]] for doc in docs]
        rerank_scores = rerank_model.predict(rerank_inputs)

        ranked_pairs = sorted(
            zip(docs, rerank_scores),
            key=lambda x: x[1],
            reverse=True
        )

        filtered_ranked = [
            (doc, score) for doc, score in ranked_pairs
            if score >= RERANK_SCORE_THRESHOLD
        ]

        ranked = filtered_ranked[:TOP_K_RERANK]

        for rank, (doc, score) in enumerate(ranked, start=1):
            all_results.append({
                "query_id": query_id,
                "chunk_id": doc["chunk_id"],
                "chunk": doc["chunk"],
                "rank": rank,
                "score": float(score),
            })

    save_jsonl(OUTPUT_RESULTS, all_results)
    print(f"[Done] Save results to {OUTPUT_RESULTS}")

if __name__ == "__main__":
    main()

d:\HocTap\ChatBot\Medical_Chatbot\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Info] Load embedding model: bkai-foundation-models/vietnamese-bi-encoder


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5582.16it/s]


[Info] Load rerank model: itdainb/PhoRanker


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5273.05it/s]
C:\Users\Windows\AppData\Local\Temp\ipykernel_8472\271002957.py:56: UserWarning: Qdrant client version 1.15.1 is incompatible with server version 1.18.2. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  client = QdrantClient(url=QDRANT_URL)
Queries:   0%|          | 0/1000 [00:00<?, ?it/s]C:\Users\Windows\AppData\Local\Temp\ipykernel_8472\271002957.py:66: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  search_hits = client.search(
[transformers] Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
[transformers] Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequenc

[Done] Save results to ./eval_search_results.jsonl


In [2]:
import json
import math
from collections import defaultdict

QRELS_FILE = "./eval_qrels.jsonl"
RESULTS_FILE = "./eval_search_results.jsonl"

def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records

def build_qrels(qrels):
    qrel = defaultdict(dict)
    for item in qrels:
        qid = item["query_id"]
        cid = item["chunk_id"]
        rel = item.get("relevance", 1)
        qrel[qid][cid] = rel
    return qrel

def build_results(results):
    ranked = defaultdict(list)
    for item in results:
        qid = item["query_id"]
        cid = item["chunk_id"]
        rank = item.get("rank")
        if rank is None:
            ranked[qid].append(cid)
        else:
            ranked[qid].append((rank, cid))
    out = {}
    for qid, items in ranked.items():
        if items and isinstance(items[0], tuple):
            items = sorted(items, key=lambda x: x[0])
            out[qid] = [cid for _, cid in items]
        else:
            out[qid] = items
    return out

def precision_at_k(preds, rels, k):
    if not preds:
        return 0.0
    topk = preds[:k]
    hits = sum(1 for cid in topk if cid in rels)
    return hits / k

def recall_at_k(preds, rels, k):
    if not rels:
        return 0.0
    topk = preds[:k]
    hits = sum(1 for cid in topk if cid in rels)
    return hits / len(rels)

def average_precision(preds, rels, k=None):
    if not rels:
        return 0.0
    if k is None:
        k = len(preds)
    hits = 0
    score = 0.0
    for idx, cid in enumerate(preds[:k], start=1):
        if cid in rels:
            hits += 1
            score += hits / idx
    return score / len(rels)

def reciprocal_rank(preds, rels):
    for idx, cid in enumerate(preds, start=1):
        if cid in rels:
            return 1.0 / idx
    return 0.0

def dcg_at_k(preds, rels, k):
    score = 0.0
    for idx, cid in enumerate(preds[:k], start=1):
        rel = 1 if cid in rels else 0
        if rel > 0:
            score += (2**rel - 1) / math.log2(idx + 1)
    return score

def idcg_at_k(rels, k):
    ideal_rels = sorted([1] * len(rels), reverse=True)
    score = 0.0
    for idx, rel in enumerate(ideal_rels[:k], start=1):
        score += (2**rel - 1) / math.log2(idx + 1)
    return score

def ndcg_at_k(preds, rels, k):
    dcg = dcg_at_k(preds, rels, k)
    idcg = idcg_at_k(rels, k)
    return dcg / idcg if idcg > 0 else 0.0

def evaluate(qrel, results, ks=(1, 3, 5, 10)):
    qids = sorted(set(qrel.keys()) & set(results.keys()))
    metrics = {f"precision@{k}": 0.0 for k in ks}
    metrics.update({f"recall@{k}": 0.0 for k in ks})
    metrics.update({f"ndcg@{k}": 0.0 for k in ks})
    metrics["map"] = 0.0
    metrics["mrr"] = 0.0

    for qid in qids:
        preds = results[qid]
        rels = qrel[qid]

        for k in ks:
            metrics[f"precision@{k}"] += precision_at_k(preds, rels, k)
            metrics[f"recall@{k}"] += recall_at_k(preds, rels, k)
            metrics[f"ndcg@{k}"] += ndcg_at_k(preds, rels, k)

        metrics["map"] += average_precision(preds, rels, max(ks))
        metrics["mrr"] += reciprocal_rank(preds, rels)

    n = len(qids) if qids else 1
    for key in metrics:
        metrics[key] /= n
    metrics["queries_evaluated"] = n
    return metrics

def main():
    qrels = load_jsonl(QRELS_FILE)
    results = load_jsonl(RESULTS_FILE)

    qrel_map = build_qrels(qrels)
    result_map = build_results(results)

    metrics = evaluate(qrel_map, result_map, ks=(1, 3, 5, 10))

    print("Evaluation results:")
    for key, value in metrics.items():
        print(f"{key}: {value:.6f}")

if __name__ == "__main__":
    main()

Evaluation results:
precision@1: 0.752764
precision@3: 0.315578
precision@5: 0.202211
precision@10: 0.101106
recall@1: 0.521617
recall@3: 0.625334
recall@5: 0.652454
recall@10: 0.652454
ndcg@1: 0.752764
ndcg@3: 0.662502
ndcg@5: 0.657537
ndcg@10: 0.652388
map: 0.584920
mrr: 0.822730
queries_evaluated: 995.000000
